> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — network evidence representation

PCSF is the interpretable baseline. GNN models should be treated as optional predictive layers with held-out validation.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
sim=vigor.simulate_genomic_dataset(n_samples=320,n_variants=1000,n_traits=8,seed=SEED,genotype_kwargs={"n_chromosomes":5,"generations":100},trait_kwargs=cfg)
X=sim.genotypes.to_pandas(); Y=sim.phenotypes.to_pandas()
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False

edges=pd.DataFrame({"source":["geneA","geneB","geneC","geneA"],"target":["geneB","geneC","geneD","geneD"],"cost":[1.,1.,1.,2.]})
prizes={"geneA":2.0,"geneD":1.5}
print(edges); print(prizes)


In [ ]:
# A small demonstration network; these names are not HS-rat gene evidence.
from scipy.sparse.csgraph import connected_components
edges=pd.DataFrame({"source":["geneA","geneB","geneC","geneD","geneE","geneA"],
                    "target":["geneB","geneC","geneD","geneE","geneF","geneF"],"cost":[1.,1.,1.,1.,1.,2.]})
prizes={"geneA":2.,"geneF":1.5}
adjacency,nodes,node_index=vigor.edge_table_to_csr(edges)
print("CSR components:",connected_components(adjacency,directed=False)[0])
result=vigor.prioritize_network(edges,prizes,method="greedy")
print("PCSF nodes:",result.nodes)
print("diffusion:",vigor.diffuse_scores(edges,prizes))
features=pd.DataFrame(np.random.default_rng(42).normal(size=(6,3)),index=nodes)
labels=pd.Series([0,1,0,1,0,1],index=nodes)
loci=pd.Series(["l1","l1","l2","l2","l3","l3"],index=nodes)
gnn=vigor.fit_heterogeneous_gnn(edges,features,labels,
    split={"held_out_loci":["l3"]},node_loci=loci,architecture="gcn",epochs=20,patience=5,random_state=42)
print("GNN:",type(gnn).__name__)
